# 04 — Model Comparison
**Loads results from:** `gp_results.pkl`, `xgb_results.pkl`, `mlp_results.pkl`

**Purpose:** Compare GP, XGBoost, and MLP side-by-side so you can make an
informed decision about which model to trust for FEM input.

**Run this notebook AFTER running 01, 02, 03.**

### What to look for:

| Metric | Meaning |
|---|---|
| LOO-CV MAE | Mean absolute error on held-out points — lower is better |
| R² | Fraction of variance explained — closer to 1 is better |
| Residual pattern | Are errors random, or clustered at specific Cr%? |
| Uncertainty bands | GP: analytic. MLP: bootstrap. XGB: none (uses MAE band). |
| Behaviour between points | Does the curve make physical sense? |


---
## Cell 1 — Imports + Load All Results

In [ ]:
DATA_PATH = '../analysis/elastic_constants_fecr.csv'
d = load_data(DATA_PATH)
df          = d['df']
X           = d['X']
X_pred      = d['X_pred']
x_pred_atoms= d['x_pred_atoms']
targets     = d['targets']
noise_gpa   = d['noise_gpa']
flagged     = d['flagged']
tier        = d['tier']


---
## Cell 2 — Performance Table

Side-by-side LOO-CV MAE and R² for all models and all targets.

In [ ]:
rows = []
for mname, mres in models.items():
    for tname in TARGETS:
        if tname in mres:
            rows.append({'Model': mname, 'Target': tname,
                         'LOO MAE (GPa)': round(mres[tname]['mae'], 2),
                         'R²': round(mres[tname]['r2'], 4)})
perf = pd.DataFrame(rows)
pivot_mae = perf.pivot(index='Target', columns='Model', values='LOO MAE (GPa)')
pivot_r2  = perf.pivot(index='Target', columns='Model', values='R²')
print('LOO-CV MAE (GPa) — lower is better:')
print(pivot_mae.to_string())
print()
print('LOO-CV R² — closer to 1 is better:')
print(pivot_r2.to_string())

In [ ]:
# Visual performance table
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['steelblue','darkorange','seagreen']
x = np.arange(len(TARGETS)); bw = 0.25

for ax, pivot, title, invert in [
    (axes[0], pivot_mae, 'LOO-CV MAE (GPa) — lower better', False),
    (axes[1], pivot_r2,  'LOO-CV R² — higher better',       False)
]:
    for i, (mname, col) in enumerate(zip(pivot.columns, colors[:len(pivot.columns)])):
        vals = [pivot.loc[t, mname] if t in pivot.index else np.nan for t in TARGETS]
        ax.bar(x + i*bw, vals, bw, label=mname, color=col, edgecolor='k', lw=0.5)
    ax.set_xticks(x + bw); ax.set_xticklabels(TARGETS)
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3, axis='y')

plt.suptitle('Model Performance Comparison — LOO-CV', fontweight='bold')
plt.tight_layout()
plt.savefig('../analysis/model_comparison_performance.png', bbox_inches='tight')
plt.show(); print('Saved: model_comparison_performance.png')

---
## Cell 3 — Prediction Overlay: All Models on One Plot

Overlay GP mean, XGBoost, and MLP predictions for each target.
Only GP uncertainty band shown (most physically meaningful).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
model_styles = {
    'GP':      {'color':'steelblue',   'ls':'-',  'lw':2.5},
    'XGBoost': {'color':'darkorange',  'ls':'--', 'lw':2.0},
    'MLP':     {'color':'seagreen',    'ls':'-.', 'lw':2.0},
}

for ax, tname in zip(axes, TARGETS):
    # GP uncertainty band (most meaningful)
    if 'GP' in models and 'std' in models['GP'][tname]:
        mu_gp  = models['GP'][tname]['mu']
        sig_gp = models['GP'][tname]['std']
        ax.fill_between(x_pred_atoms, mu_gp-sig_gp, mu_gp+sig_gp,
                        alpha=0.15, color='steelblue', label='GP ±1σ')
    # All model predictions
    for mname, mres in models.items():
        if tname not in mres: continue
        s = model_styles[mname]
        ax.plot(x_pred_atoms, mres[tname]['mu'],
                color=s['color'], ls=s['ls'], lw=s['lw'], label=mname)
    # DFT data
    ax.scatter(df['n_cr'][~flagged], targets[tname][~flagged],
               color='black', s=55, zorder=6, label='DFT')
    ax.scatter(df['n_cr'][flagged],  targets[tname][flagged],
               color='tomato', s=75, marker='D', zorder=6, label='⚠️ flagged')
    ax.set_xlabel('Cr atoms (out of 16)'); ax.set_ylabel(f'{tname} (GPa)')
    ax.set_title(f'{tname} — All Models'); ax.legend(fontsize=9)
    ax.grid(alpha=0.3); ax.set_xlim(-0.5, 16.5)

plt.suptitle('Prediction Overlay: GP vs XGBoost vs MLP', fontweight='bold')
plt.tight_layout()
plt.savefig('../analysis/model_comparison_overlay.png', bbox_inches='tight')
plt.show(); print('Saved: model_comparison_overlay.png')

---
## Cell 4 — Residual Comparison Grid

9 panels: 3 models × 3 targets. Reveals which model struggles where.

In [ ]:
fig, axes = plt.subplots(len(models), 3, figsize=(15, 4*len(models)))
if len(models) == 1: axes = axes.reshape(1, -1)

for row, (mname, mres) in enumerate(models.items()):
    for col, tname in enumerate(TARGETS):
        ax = axes[row, col]
        if tname not in mres:
            ax.text(0.5, 0.5, 'No data', ha='center', transform=ax.transAxes)
            continue
        res  = mres[tname]
        cols = [{'A':'steelblue','B':'goldenrod','C':'tomato'}[t] for t in tier]
        ax.bar(df['n_cr'].values, res['residuals'], color=cols, edgecolor='k', lw=0.5)
        ax.axhline(0, color='k', lw=1)
        ax.set_title(f'{mname} — {tname}\nMAE={res["mae"]:.2f} GPa')
        ax.set_xlabel('Cr atoms'); ax.set_ylabel('Residual (GPa)')
        ax.grid(alpha=0.3, axis='y')

plt.suptitle('LOO-CV Residuals: All Models × All Targets  (red = flagged)',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../analysis/model_comparison_residuals.png', bbox_inches='tight')
plt.show(); print('Saved: model_comparison_residuals.png')

---
## Cell 5 — Model Disagreement Map

Where do the models agree? Where do they disagree?
Large disagreement at a Cr% = that region is uncertain — avoid using it
as FEM input without additional DFT data.

**Note:** Only computed where all 3 models are available.

In [ ]:
if len(models) == 3:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    mnames = list(models.keys())
    for ax, tname in zip(axes, TARGETS):
        preds_all = np.array([models[m][tname]['mu'] for m in mnames])  # (3, 200)
        spread = preds_all.max(axis=0) - preds_all.min(axis=0)  # max - min
        ax.fill_between(x_pred_atoms, 0, spread, alpha=0.4, color=PALETTE[tname])
        ax.plot(x_pred_atoms, spread, color=PALETTE[tname], lw=2)
        ax.set_xlabel('Cr atoms (out of 16)')
        ax.set_ylabel('Max − Min prediction (GPa)')
        ax.set_title(f'{tname} — Model Disagreement')
        ax.grid(alpha=0.3); ax.set_xlim(-0.5,16.5)
    plt.suptitle('Model Disagreement (max − min across GP, XGBoost, MLP)',
                 fontweight='bold')
    plt.tight_layout()
    plt.savefig('../analysis/model_disagreement.png', bbox_inches='tight')
    plt.show(); print('Saved: model_disagreement.png')
else:
    print('Need all 3 models for disagreement map. Run 01, 02, 03 first.')

---
## Cell 6 — Recommendation Summary

Printed guidance to help you choose which model output to use for FEM.

In [ ]:
print('='*60)
print('MODEL COMPARISON SUMMARY')
print('='*60)
print()
print('LOO-CV MAE (GPa):')
print(pivot_mae.to_string())
print()
print('LOO-CV R²:')
print(pivot_r2.to_string())
print()
print('Interpretation guide:')
print('  GP      — Best for interpolation. Uncertainty is calibrated.')
print('            Recommended for FEM input (use GP mean + GP std).')
print('  XGBoost — Staircase-like predictions due to tree structure.')
print('            Good for checking GP is not overfitting trend.')
print('            No reliable uncertainty. Use MAE band only.')
print('  MLP     — Smooth predictions. May extrapolate poorly.')
print('            Bootstrap uncertainty is approximate, not calibrated.')
print('            Compare with GP — large disagreement = unreliable region.')
print()
print('Recommendation for FEM:')
print('  1. Use GP mean as the primary Cij input.')
print('  2. Use GP std to set upper/lower bound runs in FEM.')
print('  3. Flag compositions where GP/XGB/MLP disagree > 5 GPa.')
print()
print('Known caveats (do not forget):')
print('  - C11-C12 denominator 3ε: verify vs primary reference')
print('  - fe02cr14/15/16: mixed magnetic setup, down-weighted')
print('  - E[111] formula: verify vs Nye Physical Properties of Crystals')
print('  - ABAQUS card: verify *Elastic ANISOTROPIC format for your version')